# **Sentiment Analysis using NLP Pipeline & ML Models**

In [1]:
import pandas as pd
import numpy as np
import re
import string

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### **Import Dataset**

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


### **Load Dataset**

In [19]:
import os

df = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### **Data Understanding**

In [20]:
print("Shape:", df.shape)

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

df['review'][0]

Shape: (50000, 2)

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

### **NLP Preprocessing (Reusable Functions)**

In [21]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    tokens = text.split()

    # Remove stopwords + Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

### **Apply Preprocessing**

In [22]:
df['clean_review'] = df['review'].apply(clean_text)

df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


### **Encode Labels**

In [23]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

### **Train-Test Split**

In [24]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## **Feature Engineering**

### **Bag of Words (BoW)**

In [25]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

### **TF-IDF**

In [26]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## **Model Training**

### **Logistic Regression**

In [27]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

### **Naive Bayes**

In [28]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

### **Decision Tree**

In [29]:
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

y_pred_dt = dt.predict(X_test_bow)

### **Evaluation Function**

In [30]:
def evaluate(y_test, y_pred, model_name):
    print(f"\n📊 {model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

### **Evaluate Models**

In [31]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")


📊 Logistic Regression
Accuracy: 0.8852
Precision: 0.8771079666602055
Recall: 0.8979956340543759
F1 Score: 0.8874289076289469

📊 Naive Bayes
Accuracy: 0.8452
Precision: 0.8501504513540622
Recall: 0.8410398888668387
F1 Score: 0.8455706304868316

📊 Decision Tree
Accuracy: 0.7183
Precision: 0.7231820008035356
Recall: 0.7144274657670172
F1 Score: 0.7187780772686433


### **Comparison**

In [32]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
})

results.sort_values(by='F1 Score', ascending=False)

,Model,Accuracy,F1 Score
0,Logistic Regression,0.8852,0.887429
1,Naive Bayes,0.8452,0.845571
2,Decision Tree,0.7183,0.718778


## 🔍 Insights & Conclusion

- Logistic Regression performed the best due to its effectiveness in high-dimensional sparse data like TF-IDF.
- Naive Bayes performed well but assumes independence between features.
- Decision Tree performed poorly due to overfitting on sparse text data.

### Best Combination:
TF-IDF + Logistic Regression

### Key Learnings:
- Proper preprocessing improves accuracy significantly.
- TF-IDF captures importance of words better than BoW.
- Simpler models often outperform complex ones in NLP tasks.